In [1]:
# ============================================================
# Project setup
# ============================================================

import os
import sys
import json
import subprocess
from pathlib import Path

import pandas as pd

# Reproducible proxy-token measurement used only for controlled comparisons
# in Section H. Provider billing tokens continue to come from the live result JSONs.
try:
    import tiktoken
except ImportError:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "tiktoken==0.14.0"],
        check=True,
    )
    import tiktoken

REPO_URL = "https://github.com/divvy2001/A2_PE6201_GRP4.git"
repo_override = os.getenv("PE6201_REPO_ROOT")
REPO_DIR = Path(repo_override or "/content/A2_PE6201_GRP4").expanduser().resolve()

# A local override is useful for reproducibility checks outside Colab.
if repo_override:
    if not REPO_DIR.exists():
        raise FileNotFoundError(f"PE6201_REPO_ROOT does not exist: {REPO_DIR}")
else:
    # Clone on a fresh Colab runtime; otherwise refresh main with a fast-forward pull.
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    if (REPO_DIR / ".git").exists():
        subprocess.run(["git", "checkout", "main"], cwd=REPO_DIR, check=True)
        subprocess.run(["git", "pull", "--ff-only", "origin", "main"], cwd=REPO_DIR, check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

if (REPO_DIR / ".git").exists():
    CURRENT_REPO_COMMIT = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True
    ).strip()
else:
    CURRENT_REPO_COMMIT = "not_available_non_git_copy"

print("Current directory:", os.getcwd())
print("Repository ready:", os.path.exists("src"))
print("Cost-notebook runtime commit:", CURRENT_REPO_COMMIT)


Current directory: /content/A2_PE6201_GRP4
Repository ready: True
Cost-notebook runtime commit: 4dafb748f5360b65edd65c6b4443f34ee3909fa4


# 6 — Cost-to-Serve Model

**Owner:** LIANG LUWEN  
**Scope:** D6 three-layer cost model, sensitivity, break-even analysis and the four cost levers.  
**Finalisation note:** live result artifacts remain the source of billing-token measurements; proxy-token comparisons are explicitly labelled where used.


## Section A — Business Assumptions

This section defines the fixed business assumptions for Problem A:
health-insurance claim first response.

All monetary values are expressed in US dollars. These assumptions are kept
separate from measured model and evaluation data so that the cost model remains
transparent and easy to update.

In [2]:
# ============================================================
# D6 · Problem A business assumptions
# ============================================================

# All monetary values are in US dollars.
CURRENCY = "USD"

# Official A2 Problem A assumptions.
MONTHLY_VOLUME = 8_000
ASSESSOR_USD_PER_HOUR = 38.0
MINUTES_PER_ESCALATION = 12.0

# Layer 2: cost of one human fallback.
FAILURE_COST_USD = ASSESSOR_USD_PER_HOUR * MINUTES_PER_ESCALATION / 60

# Layer 3 prototype-scope assumption agreed for the team cost model.
# No dedicated monthly infrastructure, licence, monitoring, storage or
# maintenance charge is documented for the submitted prototype.
FIXED_MONTHLY_USD = 0.0
FIXED_MONTHLY_SCOPE_NOTE = (
    "Prototype-scope assumption only; not an estimate of production operating cost."
)

# Deployment caps.
DEPLOYMENT_STEP_CAP = 8
PER_RUN_BUDGET_CEILING_USD = 0.05
MONTHLY_RUN_LIMIT_PER_USER = 200
MONTHLY_USER_BUDGET_CEILING_USD = (
    PER_RUN_BUDGET_CEILING_USD * MONTHLY_RUN_LIMIT_PER_USER
)

# Final battery configuration.
V2_FROZEN_COMMIT = "74072ad"
V1_CONTROL_COMMIT = "57a21e6"  # retained team control run; limitation disclosed below
JUDGE_MODEL = "openai/gpt-4.1-mini"
EXPECTED_FINAL_RESULTS = 6
SELECTED_BASELINE_MODEL = "openai/gpt-5.6-luna"

# Required sensitivity range: measured success rate ±10 percentage points.
SENSITIVITY_SPREAD = 0.10
SENSITIVITY_STEP = 0.05

assert abs(FAILURE_COST_USD - 7.60) < 1e-9
assert abs(MONTHLY_USER_BUDGET_CEILING_USD - 10.0) < 1e-9

print(f"Currency: {CURRENCY}")
print(f"Monthly volume: {MONTHLY_VOLUME:,} claims")
print(f"Human fallback cost: US${FAILURE_COST_USD:.2f} per escalated claim")
print(f"Layer 3 fixed monthly assumption: US${FIXED_MONTHLY_USD:.2f}")
print(
    "Deployment caps:",
    f"{DEPLOYMENT_STEP_CAP} steps, US${PER_RUN_BUDGET_CEILING_USD:.2f}/run,",
    f"{MONTHLY_RUN_LIMIT_PER_USER} runs/user/month",
)
print(f"V2 frozen evaluation commit: {V2_FROZEN_COMMIT}")
print(f"Judge model: {JUDGE_MODEL}")
print(f"Sensitivity range: ±{SENSITIVITY_SPREAD:.0%}, step {SENSITIVITY_STEP:.0%}")


Currency: USD
Monthly volume: 8,000 claims
Human fallback cost: US$7.60 per escalated claim
Layer 3 fixed monthly assumption: US$0.00
Deployment caps: 8 steps, US$0.05/run, 200 runs/user/month
V2 frozen evaluation commit: 74072ad
Judge model: openai/gpt-4.1-mini
Sensitivity range: ±10%, step 5%


## Section B — Model Pricing Configuration

Model prices are kept in one place so downstream cost calculations can be
reproduced without copying token-cost numbers by hand.

The listed input/output rates below were rechecked on OpenRouter on
18 September 2026. They are standard listed rates per 1 million tokens;
cache-specific or promotional effective pricing is not used in the baseline.


In [3]:
MODEL_PRICES = {
    "anthropic/claude-haiku-4.5": {
        "input_per_1m": 1.00,
        "output_per_1m": 5.00,
        "price_checked_date": "2026-09-18",
        "price_source": "https://openrouter.ai/anthropic/claude-haiku-4.5",
    },

    "google/gemini-2.5-flash-lite": {
        "input_per_1m": 0.10,
        "output_per_1m": 0.40,
        "price_checked_date": "2026-09-18",
        "price_source": "https://openrouter.ai/google/gemini-2.5-flash-lite/pricing",
    },

    "qwen/qwen3.7-plus": {
        "input_per_1m": 0.32,
        "output_per_1m": 1.28,
        "price_checked_date": "2026-09-18",
        "price_source": "https://openrouter.ai/qwen/qwen3.7-plus",
    },

    "mistralai/mistral-small-2603": {
        "input_per_1m": 0.15,
        "output_per_1m": 0.60,
        "price_checked_date": "2026-09-18",
        "price_source": "https://openrouter.ai/mistralai/mistral-small-2603/pricing",
    },

    "openai/gpt-5.6-luna": {
        "input_per_1m": 0.20,
        "output_per_1m": 1.20,
        "price_checked_date": "2026-09-18",
        "price_source": "https://openrouter.ai/openai/gpt-5.6-luna-20260709",
    },
}

print("Pricing configured for", len(MODEL_PRICES), "model IDs.")


Pricing configured for 5 model IDs.


## Section C — Cost Functions

These functions implement the Class 5 three-layer cost-to-serve model.

Layer 1 uses measured input/output token counts and verified model list prices.
Layer 2 adds the expected human fallback cost based on measured success rate.
Layer 3 adds fixed monthly cost separately at the monthly aggregation level.

In [4]:
# ============================================================
# D6 · Core cost functions
# ============================================================

def baseline_variable_cost(
    tokens_in,
    tokens_out,
    input_price_per_1m,
    output_price_per_1m,
    retrieval_usd=0.0,
    tool_usd=0.0,
):
    """
    Layer 1 — per-task variable cost.

    Uses measured input/output token counts and model list prices.
    """
    assert tokens_in >= 0, "tokens_in cannot be negative"
    assert tokens_out >= 0, "tokens_out cannot be negative"
    assert input_price_per_1m >= 0, "input price cannot be negative"
    assert output_price_per_1m >= 0, "output price cannot be negative"

    return (
        tokens_in / 1_000_000 * input_price_per_1m
        + tokens_out / 1_000_000 * output_price_per_1m
        + retrieval_usd
        + tool_usd
    )


def expected_fallback_cost(success_rate, failure_cost_usd=FAILURE_COST_USD):
    """
    Layer 2 — expected human fallback cost per task.
    """
    assert 0.0 <= success_rate <= 1.0, \
        "success_rate must be between 0 and 1"

    return (1.0 - success_rate) * failure_cost_usd


def cost_per_successful_task(
    variable_cost_usd,
    success_rate,
    failure_cost_usd=FAILURE_COST_USD,
):
    """
    Layer 1 + Layer 2.
    Main cost-to-serve measure used in D6.
    """
    return (
        variable_cost_usd
        + expected_fallback_cost(
            success_rate,
            failure_cost_usd
        )
    )


def monthly_cost(
    cost_per_task_usd,
    volume=MONTHLY_VOLUME,
    fixed_monthly_usd=FIXED_MONTHLY_USD,
):
    """
    Layer 1 + Layer 2 at monthly volume, plus Layer 3 fixed monthly cost.
    """
    if fixed_monthly_usd is None:
        raise ValueError(
            "FIXED_MONTHLY_USD is still unresolved. "
            "Document the team assumption before calculating all-in monthly cost."
        )

    return (
        cost_per_task_usd * volume
        + fixed_monthly_usd
    )


def break_even_success_rate(
    cheap_variable_cost_usd,
    expensive_total_cost_usd,
    failure_cost_usd=FAILURE_COST_USD,
):
    """
    Success rate required for the cheaper model to match
    the expensive model's full cost-to-serve.
    """
    assert failure_cost_usd > 0, \
        "failure_cost_usd must be greater than zero"

    p = 1.0 - (
        expensive_total_cost_usd
        - cheap_variable_cost_usd
    ) / failure_cost_usd

    # Keep result inside a valid probability range.
    return max(0.0, min(1.0, p))


print("Core D6 cost functions loaded successfully.")

Core D6 cost functions loaded successfully.


## Section D — Evaluation Results and Cost Ledger

This section loads the final standardised evaluation artifacts in
`outputs/results/` and converts every executed trial into one row of the D6
cost ledger.

The final result directory contains five v2 live-model runs and one Gemini v1
control. Older/raw result files are stored under `outputs/results/archive_pre_final/`
and are intentionally excluded from the cost model.


### D.1 — Final Evaluation JSON Ingestion

Only files matching `results__*.json` in the root result directory are ingested.
The configuration key combines model, prompt version and execution mode, so the
Gemini v1 control remains separate from Gemini v2.


In [5]:
from pathlib import Path

RESULT_DIR = Path("outputs/results")

result_files = sorted(RESULT_DIR.glob("results__*.json"))

print(f"Found {len(result_files)} final result files:")

for path in result_files:
    print("-", path.name)

assert len(result_files) == EXPECTED_FINAL_RESULTS, (
    f"Expected {EXPECTED_FINAL_RESULTS} final result files, "
    f"found {len(result_files)}."
)


Found 6 final result files:
- results__anthropic-claude-haiku-4-5__v2__parallel.json
- results__google-gemini-2-5-flash-lite__v1__parallel.json
- results__google-gemini-2-5-flash-lite__v2__parallel.json
- results__mistralai-mistral-small-2603__v2__parallel.json
- results__openai-gpt-5-6-luna__v2__parallel.json
- results__qwen-qwen3-7-plus__v2__parallel.json


In [6]:
import json

for path in result_files:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    summary = data.get("summary", {})

    print("\n" + "=" * 70)
    print("File:", path.name)
    print("Model:", data.get("model"))
    print("Prompt:", data.get("prompt_version"))
    print("Autonomy:", data.get("autonomy"))
    print("Parallel:", data.get("parallel_tools"))
    print("Cases:", summary.get("total_cases"))
    print("Trials:", summary.get("total_trials"))
    print("Passed cases:", summary.get("passed_cases"))
    print("Agent tokens in:", summary.get("tokens_in"))
    print("Agent tokens out:", summary.get("tokens_out"))


File: results__anthropic-claude-haiku-4-5__v2__parallel.json
Model: anthropic/claude-haiku-4.5
Prompt: v2
Autonomy: act
Parallel: True
Cases: 41
Trials: 59
Passed cases: 21
Agent tokens in: 1019010
Agent tokens out: 89831

File: results__google-gemini-2-5-flash-lite__v1__parallel.json
Model: google/gemini-2.5-flash-lite
Prompt: v1
Autonomy: act
Parallel: True
Cases: 41
Trials: 59
Passed cases: 8
Agent tokens in: 727375
Agent tokens out: 35381

File: results__google-gemini-2-5-flash-lite__v2__parallel.json
Model: google/gemini-2.5-flash-lite
Prompt: v2
Autonomy: act
Parallel: True
Cases: 41
Trials: 59
Passed cases: 13
Agent tokens in: 890136
Agent tokens out: 48581

File: results__mistralai-mistral-small-2603__v2__parallel.json
Model: mistralai/mistral-small-2603
Prompt: v2
Autonomy: act
Parallel: True
Cases: 41
Trials: 59
Passed cases: 8
Agent tokens in: 735607
Agent tokens out: 119417

File: results__openai-gpt-5-6-luna__v2__parallel.json
Model: openai/gpt-5.6-luna
Prompt: v2
Autonom

In [7]:
import pandas as pd

rows = []

for path in result_files:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    for case in data["results"]:
        for trial in case["trials"]:
            run = trial["result"]

            rows.append({
                "source_file": path.name,
                "run_id": run.get("run_id"),
                "case_id": case.get("case_id"),
                "family": case.get("family"),
                "negative": case.get("negative"),
                "trial": trial.get("trial"),

                "model": run.get("model"),
                "prompt_version": run.get("prompt_version"),
                "backend": run.get("backend"),
                "mode": run.get("mode"),
                "autonomy": run.get("autonomy"),

                "status": run.get("status"),
                "passed": trial.get("passed"),
                "l1_passed": trial.get("l1_passed"),
                "l2_passed": trial.get("l2_passed"),

                "turns": run.get("turns"),
                "tool_calls": run.get("tool_calls"),
                "tokens_in": run.get("tokens_in"),
                "tokens_out": run.get("tokens_out"),
                "reported_cost_usd": run.get("cost_usd"),
                "latency_ms": run.get("latency_ms"),
            })

final_raw_ledger_df = pd.DataFrame(rows)

print("Ledger rows:", len(final_raw_ledger_df))
print("Models:", final_raw_ledger_df["model"].nunique())

display(final_raw_ledger_df.head())

Ledger rows: 354
Models: 5


,source_file,run_id,case_id,family,negative,trial,model,prompt_version,backend,mode,...,status,passed,l1_passed,l2_passed,turns,tool_calls,tokens_in,tokens_out,reported_cost_usd,latency_ms
0,results__anthropic-claude-haiku-4-5__v2__paral...,f504d19e-d0fb-4564-8b56-328ec9868dfb,CLM-8842,partly_payable,False,1,anthropic/claude-haiku-4.5,v2,LiveBackend,parallel,...,halted,False,False,False,12,9,36511,3257,0.0,25858.274586
1,results__anthropic-claude-haiku-4-5__v2__paral...,0c5fb22b-bc03-4ef7-a090-e683c2ed1f3a,CLM-8850,single_line_short_run,False,1,anthropic/claude-haiku-4.5,v2,LiveBackend,parallel,...,completed,True,True,True,7,6,16206,1419,0.0,13426.303775
2,results__anthropic-claude-haiku-4-5__v2__paral...,f59ee59f-f8d7-468c-8c0f-0ff62ea6842c,CLM-8861,preauth_present_and_valid,False,1,anthropic/claude-haiku-4.5,v2,LiveBackend,parallel,...,completed,True,True,True,9,8,23032,2456,0.0,20104.501022
3,results__anthropic-claude-haiku-4-5__v2__paral...,7cbff19f-fec9-4200-a561-d40c534c5c5c,CLM-8874,non_panel_hospital,False,1,anthropic/claude-haiku-4.5,v2,LiveBackend,parallel,...,completed,False,True,False,6,6,14406,1703,0.0,13661.657434
4,results__anthropic-claude-haiku-4-5__v2__paral...,7efc4acb-21fc-4b49-b51a-9c01e46d6513,CLM-8925,annual_limit_exceeded,True,1,anthropic/claude-haiku-4.5,v2,LiveBackend,parallel,...,completed,False,True,False,4,2,8045,744,0.0,7607.886311


In [8]:
# Aggregate trial-level measurements by evaluation configuration.
final_raw_ledger_df["config_id"] = (
    final_raw_ledger_df["model"].astype(str)
    + " | "
    + final_raw_ledger_df["prompt_version"].astype(str)
    + " | "
    + final_raw_ledger_df["mode"].astype(str)
)

validation_summary = (
    final_raw_ledger_df
    .groupby(
        ["config_id", "model", "prompt_version", "mode"],
        as_index=False,
    )
    .agg(
        trials=("run_id", "count"),
        tokens_in=("tokens_in", "sum"),
        tokens_out=("tokens_out", "sum"),
        passed_trials=("passed", "sum"),
        avg_turns=("turns", "mean"),
        avg_tool_calls=("tool_calls", "mean"),
    )
)

validation_summary["trial_pass_rate"] = (
    validation_summary["passed_trials"]
    / validation_summary["trials"]
)

display(validation_summary)

,config_id,model,prompt_version,mode,trials,tokens_in,tokens_out,passed_trials,avg_turns,avg_tool_calls,trial_pass_rate
0,anthropic/claude-haiku-4.5 | v2 | parallel,anthropic/claude-haiku-4.5,v2,parallel,59,1019010,89831,27,7.050847,5.491525,0.457627
1,google/gemini-2.5-flash-lite | v1 | parallel,google/gemini-2.5-flash-lite,v1,parallel,59,727375,35381,10,6.338983,5.779661,0.169492
2,google/gemini-2.5-flash-lite | v2 | parallel,google/gemini-2.5-flash-lite,v2,parallel,59,890136,48581,17,6.915254,6.372881,0.288136
3,mistralai/mistral-small-2603 | v2 | parallel,mistralai/mistral-small-2603,v2,parallel,59,735607,119417,9,5.745763,5.559322,0.152542
4,openai/gpt-5.6-luna | v2 | parallel,openai/gpt-5.6-luna,v2,parallel,59,620170,67763,35,5.627119,5.406780,0.593220
5,qwen/qwen3.7-plus | v2 | parallel,qwen/qwen3.7-plus,v2,parallel,59,492328,90010,28,4.322034,5.491525,0.474576


In [9]:
# Build one case-level evaluation summary row per result artifact.
case_summary_rows = []

for path in result_files:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    summary = data["summary"]
    model = data["model"]
    prompt_version = data.get("prompt_version")
    mode = "parallel" if data.get("parallel_tools") else "sequential"

    config_id = f"{model} | {prompt_version} | {mode}"

    case_summary_rows.append({
        "source_file": path.name,
        "config_id": config_id,
        "model": model,
        "prompt_version": prompt_version,
        "mode": mode,
        "cases": summary["total_cases"],
        "passed_cases": summary["passed_cases"],
        "case_pass_rate": summary["case_pass_rate"],
        "ordinary_cases": summary["ordinary_cases"],
        "ordinary_passed": summary["ordinary_passed"],
        "ordinary_pass_rate": summary["ordinary_pass_rate"],
        "negative_cases": summary["negative_cases"],
        "negative_passed": summary["negative_passed"],
        "negative_pass_rate": summary["negative_pass_rate"],
    })

case_summary_df = pd.DataFrame(case_summary_rows)

model_eval_summary_df = case_summary_df.merge(
    validation_summary[
        [
            "config_id",
            "trials",
            "passed_trials",
            "trial_pass_rate",
            "tokens_in",
            "tokens_out",
            "avg_turns",
            "avg_tool_calls",
        ]
    ],
    on="config_id",
    how="left",
)

display(model_eval_summary_df)

,source_file,config_id,model,prompt_version,mode,cases,passed_cases,case_pass_rate,ordinary_cases,ordinary_passed,...,negative_cases,negative_passed,negative_pass_rate,trials,passed_trials,trial_pass_rate,tokens_in,tokens_out,avg_turns,avg_tool_calls
0,results__anthropic-claude-haiku-4-5__v2__paral...,anthropic/claude-haiku-4.5 | v2 | parallel,anthropic/claude-haiku-4.5,v2,parallel,41,21,0.512195,32,18,...,9,3,0.333333,59,27,0.457627,1019010,89831,7.050847,5.491525
1,results__google-gemini-2-5-flash-lite__v1__par...,google/gemini-2.5-flash-lite | v1 | parallel,google/gemini-2.5-flash-lite,v1,parallel,41,8,0.195122,32,7,...,9,1,0.111111,59,10,0.169492,727375,35381,6.338983,5.779661
2,results__google-gemini-2-5-flash-lite__v2__par...,google/gemini-2.5-flash-lite | v2 | parallel,google/gemini-2.5-flash-lite,v2,parallel,41,13,0.317073,32,11,...,9,2,0.222222,59,17,0.288136,890136,48581,6.915254,6.372881
3,results__mistralai-mistral-small-2603__v2__par...,mistralai/mistral-small-2603 | v2 | parallel,mistralai/mistral-small-2603,v2,parallel,41,8,0.195122,32,8,...,9,0,0.000000,59,9,0.152542,735607,119417,5.745763,5.559322
4,results__openai-gpt-5-6-luna__v2__parallel.json,openai/gpt-5.6-luna | v2 | parallel,openai/gpt-5.6-luna,v2,parallel,41,29,0.707317,32,26,...,9,3,0.333333,59,35,0.593220,620170,67763,5.627119,5.406780
5,results__qwen-qwen3-7-plus__v2__parallel.json,qwen/qwen3.7-plus | v2 | parallel,qwen/qwen3.7-plus,v2,parallel,41,23,0.560976,32,21,...,9,2,0.222222,59,28,0.474576,492328,90010,4.322034,5.491525


## Section E — Cost-to-Serve Results

This section combines measured token usage, model list prices, and measured
evaluation success rates to calculate Layer 1 model cost, Layer 2 expected
human fallback cost, and monthly cost before Layer 3 fixed cost.

### E.1 — Layer 1: Model Variable Cost

In [10]:
def add_pricing_and_layer1(model_summary_df, price_config):
    df = model_summary_df.copy()

    df["input_price_per_1m"] = df["model"].map(
        lambda m: price_config[m]["input_per_1m"]
    )

    df["output_price_per_1m"] = df["model"].map(
        lambda m: price_config[m]["output_per_1m"]
    )

    df["price_checked_date"] = df["model"].map(
        lambda m: price_config[m]["price_checked_date"]
    )

    df["price_source"] = df["model"].map(
        lambda m: price_config[m]["price_source"]
    )

    df["layer1_total_eval_cost_usd"] = (
        df["tokens_in"] / 1_000_000 * df["input_price_per_1m"]
        +
        df["tokens_out"] / 1_000_000 * df["output_price_per_1m"]
    )

    df["layer1_cost_per_trial_usd"] = (
        df["layer1_total_eval_cost_usd"] / df["trials"]
    )

    return df


model_cost_base_df = add_pricing_and_layer1(
    model_eval_summary_df,
    MODEL_PRICES
)

display(
    model_cost_base_df[
        [
            "model",
            "tokens_in",
            "tokens_out",
            "input_price_per_1m",
            "output_price_per_1m",
            "layer1_total_eval_cost_usd",
            "layer1_cost_per_trial_usd",
        ]
    ]
)

,model,tokens_in,tokens_out,input_price_per_1m,output_price_per_1m,layer1_total_eval_cost_usd,layer1_cost_per_trial_usd
0,anthropic/claude-haiku-4.5,1019010,89831,1.00,5.00,1.468165,0.024884
1,google/gemini-2.5-flash-lite,727375,35381,0.10,0.40,0.086890,0.001473
2,google/gemini-2.5-flash-lite,890136,48581,0.10,0.40,0.108446,0.001838
3,mistralai/mistral-small-2603,735607,119417,0.15,0.60,0.181991,0.003085
4,openai/gpt-5.6-luna,620170,67763,0.20,1.20,0.205350,0.003481
5,qwen/qwen3.7-plus,492328,90010,0.32,1.28,0.272758,0.004623


### E.2 — Layer 2: Expected Human Fallback Cost

In [11]:
# ============================================================
# Layer 2 · Expected human fallback cost
# ============================================================

model_cost_df = model_cost_base_df.copy()

# Baseline success rate for cost-to-serve:
# measured pass rate across all executed evaluation trials
model_cost_df["success_rate"] = model_cost_df["trial_pass_rate"]

model_cost_df["layer2_fallback_cost_per_trial_usd"] = (
    (1 - model_cost_df["success_rate"])
    * FAILURE_COST_USD
)

model_cost_df["cost_to_serve_per_trial_usd"] = (
    model_cost_df["layer1_cost_per_trial_usd"]
    + model_cost_df["layer2_fallback_cost_per_trial_usd"]
)

model_cost_df["role"] = model_cost_df["prompt_version"].map(
    lambda p: "v1_control" if p == "v1" else "v2_battery"
)

model_cost_df["selected_baseline"] = (
    (model_cost_df["model"] == SELECTED_BASELINE_MODEL)
    & (model_cost_df["prompt_version"] == "v2")
)

display(
    model_cost_df[
        [
            "role",
            "model",
            "prompt_version",
            "trial_pass_rate",
            "layer1_cost_per_trial_usd",
            "layer2_fallback_cost_per_trial_usd",
            "cost_to_serve_per_trial_usd",
            "selected_baseline",
        ]
    ].sort_values(
        ["role", "cost_to_serve_per_trial_usd"],
        ascending=[False, True],
    )
)


,role,model,prompt_version,trial_pass_rate,layer1_cost_per_trial_usd,layer2_fallback_cost_per_trial_usd,cost_to_serve_per_trial_usd,selected_baseline
4,v2_battery,openai/gpt-5.6-luna,v2,0.593220,0.003481,3.091525,3.095006,True
5,v2_battery,qwen/qwen3.7-plus,v2,0.474576,0.004623,3.993220,3.997843,False
0,v2_battery,anthropic/claude-haiku-4.5,v2,0.457627,0.024884,4.122034,4.146918,False
2,v2_battery,google/gemini-2.5-flash-lite,v2,0.288136,0.001838,5.410169,5.412008,False
3,v2_battery,mistralai/mistral-small-2603,v2,0.152542,0.003085,6.440678,6.443763,False
1,v1_control,google/gemini-2.5-flash-lite,v1,0.169492,0.001473,6.311864,6.313337,False


### E.3 — Monthly Cost Including Layer 3


In [12]:
# ============================================================
# Monthly cost including Layer 3 fixed monthly cost
# ============================================================

model_cost_df["monthly_volume"] = MONTHLY_VOLUME
model_cost_df["monthly_cost_before_fixed_usd"] = (
    model_cost_df["cost_to_serve_per_trial_usd"] * model_cost_df["monthly_volume"]
)
model_cost_df["fixed_monthly_usd"] = FIXED_MONTHLY_USD
model_cost_df["monthly_all_in_usd"] = (
    model_cost_df["monthly_cost_before_fixed_usd"] + FIXED_MONTHLY_USD
)

display(
    model_cost_df[
        [
            "role",
            "model",
            "prompt_version",
            "cost_to_serve_per_trial_usd",
            "monthly_volume",
            "monthly_cost_before_fixed_usd",
            "fixed_monthly_usd",
            "monthly_all_in_usd",
        ]
    ].sort_values(
        ["role", "monthly_all_in_usd"],
        ascending=[False, True],
    )
)


,role,model,prompt_version,cost_to_serve_per_trial_usd,monthly_volume,monthly_cost_before_fixed_usd,fixed_monthly_usd,monthly_all_in_usd
4,v2_battery,openai/gpt-5.6-luna,v2,3.095006,8000,24760.047403,0.0,24760.047403
5,v2_battery,qwen/qwen3.7-plus,v2,3.997843,8000,31982.746815,0.0,31982.746815
0,v2_battery,anthropic/claude-haiku-4.5,v2,4.146918,8000,33175.344407,0.0,33175.344407
2,v2_battery,google/gemini-2.5-flash-lite,v2,5.412008,8000,43296.060475,0.0,43296.060475
3,v2_battery,mistralai/mistral-small-2603,v2,6.443763,8000,51550.100508,0.0,51550.100508
1,v1_control,google/gemini-2.5-flash-lite,v1,6.313337,8000,50506.696936,0.0,50506.696936


### E.4 — Success-Rate Convention and Diagnostic

In [13]:
# ============================================================
# Compare trial-level vs case-level success-rate assumptions
# ============================================================

success_rate_comparison_df = model_cost_base_df.copy()

# Trial-level baseline
success_rate_comparison_df["trial_success_rate"] = (
    success_rate_comparison_df["trial_pass_rate"]
)

success_rate_comparison_df["trial_layer2_usd"] = (
    (1 - success_rate_comparison_df["trial_success_rate"])
    * FAILURE_COST_USD
)

success_rate_comparison_df["trial_cost_to_serve_usd"] = (
    success_rate_comparison_df["layer1_cost_per_trial_usd"]
    + success_rate_comparison_df["trial_layer2_usd"]
)

# Case-level alternative
success_rate_comparison_df["case_success_rate"] = (
    success_rate_comparison_df["case_pass_rate"]
)

success_rate_comparison_df["case_layer2_usd"] = (
    (1 - success_rate_comparison_df["case_success_rate"])
    * FAILURE_COST_USD
)

success_rate_comparison_df["case_cost_to_serve_usd"] = (
    success_rate_comparison_df["layer1_cost_per_trial_usd"]
    + success_rate_comparison_df["case_layer2_usd"]
)

display(
    success_rate_comparison_df[
        [
            "model",
            "trial_success_rate",
            "trial_cost_to_serve_usd",
            "case_success_rate",
            "case_cost_to_serve_usd",
        ]
    ]
)

,model,trial_success_rate,trial_cost_to_serve_usd,case_success_rate,case_cost_to_serve_usd
0,anthropic/claude-haiku-4.5,0.457627,4.146918,0.512195,3.732201
1,google/gemini-2.5-flash-lite,0.169492,6.313337,0.195122,6.118546
2,google/gemini-2.5-flash-lite,0.288136,5.412008,0.317073,5.192082
3,mistralai/mistral-small-2603,0.152542,6.443763,0.195122,6.120158
4,openai/gpt-5.6-luna,0.593220,3.095006,0.707317,2.227871
5,qwen/qwen3.7-plus,0.474576,3.997843,0.560976,3.341208


**Success-rate convention.** The baseline cost-to-serve model uses the
measured trial-level pass rate, consistent with the team reporting framework,
which reports passed trials over total trials. Case-level pass rate is retained
as an alternative diagnostic because negative cases were intentionally run
three trials while ordinary cases were run once. Therefore, the baseline
success rate reflects the evaluation mix and should not be interpreted as an
estimate of the production claim distribution.

### E.5 — Layer 3 Assumption

Layer 3 is set to **US$0/month for the submitted prototype**. No dedicated
monthly infrastructure, software licence, monitoring, storage or maintenance
charge is documented for the current prototype, so the model does not invent a
vendor or labour estimate. This is an explicit modelling assumption and should
not be interpreted as an estimate of production operating cost.


## Section F — Sensitivity Analysis

Success rate is an estimated quantity, so the cost model does not report only a
single point estimate. For each evaluation configuration, cost-to-serve is
recomputed across the measured success rate ±10 percentage points, in
5-percentage-point steps.

In [14]:
# ============================================================
# D6 · Sensitivity analysis
# ============================================================

def build_sensitivity_table(
    variable_cost_usd,
    success_rate,
    failure_cost_usd=FAILURE_COST_USD,
    spread=SENSITIVITY_SPREAD,
    step=SENSITIVITY_STEP,
    configuration=None,
):
    """
    Recalculate Layer 2 and cost-to-serve across
    success rate ±10 percentage points.
    """

    assert 0.0 <= success_rate <= 1.0, \
        "success_rate must be between 0 and 1"

    lower = max(0.0, success_rate - spread)
    upper = min(1.0, success_rate + spread)

    rates = []
    current = lower

    while current <= upper + 1e-9:
        rates.append(round(current, 10))
        current += step

    rows = []

    for rate in rates:
        fallback = expected_fallback_cost(
            success_rate=rate,
            failure_cost_usd=failure_cost_usd,
        )

        total = cost_per_successful_task(
            variable_cost_usd=variable_cost_usd,
            success_rate=rate,
            failure_cost_usd=failure_cost_usd,
        )

        rows.append(
            {
                "configuration": configuration,
                "success_rate": rate,
                "baseline_variable_cost_usd": variable_cost_usd,
                "expected_fallback_cost_usd": fallback,
                "cost_to_serve_usd": total,
                "is_measured_rate": abs(rate - success_rate) < 1e-9,
            }
        )

    return pd.DataFrame(rows)

### F.1 — Formal Sensitivity Analysis Using Measured Live-Run Results

In [15]:
# ============================================================
# Formal sensitivity analysis: success rate ±10 percentage points
# ============================================================

sensitivity_tables = []

for _, row in model_cost_df.iterrows():

    table = build_sensitivity_table(
        variable_cost_usd=row["layer1_cost_per_trial_usd"],
        success_rate=row["trial_pass_rate"],
        failure_cost_usd=FAILURE_COST_USD,
        spread=SENSITIVITY_SPREAD,
        step=SENSITIVITY_STEP,
        configuration=row["config_id"],
    )

    sensitivity_tables.append(table)

final_sensitivity_df = pd.concat(
    sensitivity_tables,
    ignore_index=True
)

display(final_sensitivity_df)

,configuration,success_rate,baseline_variable_cost_usd,expected_fallback_cost_usd,cost_to_serve_usd,is_measured_rate
0,anthropic/claude-haiku-4.5 | v2 | parallel,0.357627,0.024884,4.882034,4.906918,False
1,anthropic/claude-haiku-4.5 | v2 | parallel,0.407627,0.024884,4.502034,4.526918,False
2,anthropic/claude-haiku-4.5 | v2 | parallel,0.457627,0.024884,4.122034,4.146918,True
3,anthropic/claude-haiku-4.5 | v2 | parallel,0.507627,0.024884,3.742034,3.766918,False
4,anthropic/claude-haiku-4.5 | v2 | parallel,0.557627,0.024884,3.362034,3.386918,False
5,google/gemini-2.5-flash-lite | v1 | parallel,0.069492,0.001473,7.071864,7.073337,False
6,google/gemini-2.5-flash-lite | v1 | parallel,0.119492,0.001473,6.691864,6.693337,False
7,google/gemini-2.5-flash-lite | v1 | parallel,0.169492,0.001473,6.311864,6.313337,True
8,google/gemini-2.5-flash-lite | v1 | parallel,0.219492,0.001473,5.931864,5.933337,False
9,google/gemini-2.5-flash-lite | v1 | parallel,0.269492,0.001473,5.551864,5.553337,False


### F.2 — Sensitivity Summary

In [16]:
# ============================================================
# E.2 · Compact sensitivity summary for reporting
# ============================================================

sensitivity_summary_rows = []

for model in final_sensitivity_df["configuration"].unique():

    model_df = final_sensitivity_df[
        final_sensitivity_df["configuration"] == model
    ].sort_values("success_rate")

    measured_row = model_df[
        model_df["is_measured_rate"] == True
    ].iloc[0]

    low_row = model_df.iloc[0]
    high_row = model_df.iloc[-1]

    sensitivity_summary_rows.append({
        "configuration": model,

        "low_success_rate": low_row["success_rate"],
        "low_cost_to_serve_usd": low_row["cost_to_serve_usd"],

        "measured_success_rate": measured_row["success_rate"],
        "measured_cost_to_serve_usd": measured_row["cost_to_serve_usd"],

        "high_success_rate": high_row["success_rate"],
        "high_cost_to_serve_usd": high_row["cost_to_serve_usd"],
    })

sensitivity_summary_df = pd.DataFrame(
    sensitivity_summary_rows
)

display(sensitivity_summary_df)

,configuration,low_success_rate,low_cost_to_serve_usd,measured_success_rate,measured_cost_to_serve_usd,high_success_rate,high_cost_to_serve_usd
0,anthropic/claude-haiku-4.5 | v2 | parallel,0.357627,4.906918,0.457627,4.146918,0.557627,3.386918
1,google/gemini-2.5-flash-lite | v1 | parallel,0.069492,7.073337,0.169492,6.313337,0.269492,5.553337
2,google/gemini-2.5-flash-lite | v2 | parallel,0.188136,6.172008,0.288136,5.412008,0.388136,4.652008
3,mistralai/mistral-small-2603 | v2 | parallel,0.052542,7.203763,0.152542,6.443763,0.252542,5.683763
4,openai/gpt-5.6-luna | v2 | parallel,0.493220,3.855006,0.593220,3.095006,0.693220,2.335006
5,qwen/qwen3.7-plus | v2 | parallel,0.374576,4.757843,0.474576,3.997843,0.574576,3.237843


### F.3 — Sensitivity Interpretation

Human fallback remains the dominant component of cost-to-serve across the
±10 percentage-point success-rate range. This conclusion is robust because
Layer 2 remains several dollars per task, while Layer 1 model cost remains only
a few cents or less.

However, the relative cost ranking between models is not robust. Within the
tested sensitivity range, changes in success rate are large enough to reverse
which model has the lowest overall cost-to-serve. Therefore, model selection
should not be based on token price alone.

## Section G — Break-even Analysis

Break-even analysis asks how successful the lowest-Layer-1-cost v2 model must
be to match the measured cost-to-serve of the selected v2 baseline.

The lowest-Layer-1-cost v2 model is Gemini 2.5 Flash Lite. The selected baseline
is GPT-5.6 Luna because it has the lowest measured cost-to-serve among the five
final v2 configurations.


In [17]:
# ============================================================
# D6 · Break-even analysis
# ============================================================

def build_break_even_result(
    cheap_model,
    cheap_variable_cost_usd,
    expensive_model,
    expensive_variable_cost_usd,
    expensive_success_rate,
    failure_cost_usd=FAILURE_COST_USD,
):
    """
    Calculate the minimum success rate required for the cheaper model
    to match the expensive model's measured cost-to-serve.
    """

    expensive_total_cost = cost_per_successful_task(
        variable_cost_usd=expensive_variable_cost_usd,
        success_rate=expensive_success_rate,
        failure_cost_usd=failure_cost_usd,
    )

    required_success_rate = break_even_success_rate(
        cheap_variable_cost_usd=cheap_variable_cost_usd,
        expensive_total_cost_usd=expensive_total_cost,
        failure_cost_usd=failure_cost_usd,
    )

    return {
        "cheap_model": cheap_model,
        "cheap_variable_cost_usd": cheap_variable_cost_usd,
        "expensive_model": expensive_model,
        "expensive_variable_cost_usd": expensive_variable_cost_usd,
        "expensive_success_rate": expensive_success_rate,
        "expensive_cost_to_serve_usd": expensive_total_cost,
        "failure_cost_usd": failure_cost_usd,
        "break_even_success_rate": required_success_rate,
    }

### G.1 — Formal Break-even Analysis Using Measured Live-Run Results

In [18]:
# ============================================================
# G.1 · Formal break-even analysis
# ============================================================

cheap_model = "google/gemini-2.5-flash-lite"
benchmark_model = SELECTED_BASELINE_MODEL

cheap_row = model_cost_df[
    (model_cost_df["model"] == cheap_model)
    & (model_cost_df["prompt_version"] == "v2")
].iloc[0]

benchmark_row = model_cost_df[
    (model_cost_df["model"] == benchmark_model)
    & (model_cost_df["prompt_version"] == "v2")
].iloc[0]

formal_break_even_result = build_break_even_result(
    cheap_model=cheap_model,
    cheap_variable_cost_usd=cheap_row["layer1_cost_per_trial_usd"],

    expensive_model=benchmark_model,
    expensive_variable_cost_usd=benchmark_row["layer1_cost_per_trial_usd"],
    expensive_success_rate=benchmark_row["trial_pass_rate"],

    failure_cost_usd=FAILURE_COST_USD,
)

formal_break_even_df = pd.DataFrame(
    [formal_break_even_result]
)

formal_break_even_df["cheap_measured_success_rate"] = cheap_row["trial_pass_rate"]
formal_break_even_df["measured_minus_break_even_pp"] = (
    (
        formal_break_even_df["cheap_measured_success_rate"]
        - formal_break_even_df["break_even_success_rate"]
    )
    * 100
)

display(formal_break_even_df)


,cheap_model,cheap_variable_cost_usd,expensive_model,expensive_variable_cost_usd,expensive_success_rate,expensive_cost_to_serve_usd,failure_cost_usd,break_even_success_rate,cheap_measured_success_rate,measured_minus_break_even_pp
0,google/gemini-2.5-flash-lite,0.001838,openai/gpt-5.6-luna,0.003481,0.59322,3.095006,7.6,0.593004,0.288136,-30.486864


### G.2 — Break-even Interpretation

Gemini 2.5 Flash Lite has the lowest measured Layer 1 cost among the final v2
models, but it would need a trial-level success rate of about **59.30%** to match
GPT-5.6 Luna's measured cost-to-serve of about **US$3.095 per task**.

Gemini's measured v2 trial-level success rate is **28.81%**, about **30.49
percentage points below** that threshold. Its lower token price therefore does
not offset the additional expected human fallback cost.

Under the assumed US$7.60 failure-handling cost, measured success rate has a
much larger economic effect than small differences in token price.


## Section H — Four Cost Levers

The cost analysis tracks four levers: tool-block size, turn count,
observation size, and success rate. Each comparison uses a consistent
measurement method and clearly states whether the evidence is live,
scripted, or reconstructed for analysis.

In [19]:
# ============================================================
# D6 · Four cost levers — evidence framework
# ============================================================

LEVER_MEASUREMENTS = {
    "tool_block_size": {
        "status": "READY_WITH_RECONSTRUCTION_NOTE",
        "reconstructed_before_tokens": None,
        "final_v2_tokens": None,
        "reconstructed_before_tool_count": None,
        "final_tool_count": None,
        "source": "D2(a) documented consolidation + reconstructed pre-consolidation descriptor block",
    },
    "turn_count": {
        "status": "READY_WITH_SCOPE_NOTE",
        "strict_sequential_avg_turns": None,
        "parallel_avg_turns": None,
        "strict_sequential_proxy_input_tokens": None,
        "parallel_proxy_input_tokens": None,
        "strict_sequential_pass_rate": None,
        "parallel_pass_rate": None,
        "source": "Controlled scripted D2(c) replay",
    },
    "observation_size": {
        "status": "READY_WITH_COMMIT_NOTE",
        "v1_avg_tokens_per_trial": None,
        "v2_avg_tokens_per_trial": None,
        "source": "Retained Gemini v1/v2 live traces; compact JSON observation messages",
    },
    "success_rate": {
        "status": "READY_WITH_COMMIT_NOTE",
        "model_results": None,
        "source": "D4/D5 retained Gemini v1/v2 live results",
    },
}


def reduction_pct(before, after):
    if before is None or after is None:
        return None
    if before == 0:
        raise ValueError("before value cannot be zero")
    return (before - after) / before


def lever_measurement_status(measurements):
    return pd.DataFrame(
        [
            {
                "lever": lever,
                "status": values["status"],
                "source": values["source"],
            }
            for lever, values in measurements.items()
        ]
    )

### H.1 — Success-Rate Lever: Retained Gemini v1→v2 Comparison

The same Gemini model is used for the v1→v2 success-rate comparison. The v1
control was retained from commit `57a21e6`, while the final v2 result was run on
`74072ad`. The comparison is therefore useful measured evidence but is **not
claimed as a perfect same-commit control**; the commit difference is disclosed
as a limitation.


In [20]:
success_rate_measurements_df = (
    model_eval_summary_df[
        [
            "config_id",
            "model",
            "prompt_version",
            "mode",
            "trials",
            "passed_trials",
            "trial_pass_rate",
            "cases",
            "passed_cases",
            "case_pass_rate",
        ]
    ]
    .copy()
)

display(success_rate_measurements_df)

gemini_control_df = success_rate_measurements_df[
    success_rate_measurements_df["model"]
    == "google/gemini-2.5-flash-lite"
].sort_values("prompt_version")

gemini_v1 = gemini_control_df[
    gemini_control_df["prompt_version"] == "v1"
].iloc[0]

gemini_v2 = gemini_control_df[
    gemini_control_df["prompt_version"] == "v2"
].iloc[0]

success_lever_df = pd.DataFrame(
    [
        {
            "model": "google/gemini-2.5-flash-lite",
            "before_prompt": "v1",
            "before_passed_trials": gemini_v1["passed_trials"],
            "before_trials": gemini_v1["trials"],
            "before_trial_pass_rate": gemini_v1["trial_pass_rate"],
            "after_prompt": "v2",
            "after_passed_trials": gemini_v2["passed_trials"],
            "after_trials": gemini_v2["trials"],
            "after_trial_pass_rate": gemini_v2["trial_pass_rate"],
            "change_percentage_points": (
                gemini_v2["trial_pass_rate"]
                - gemini_v1["trial_pass_rate"]
            )
            * 100,
        }
    ]
)

LEVER_MEASUREMENTS["success_rate"]["model_results"] = (
    success_lever_df.to_dict("records")
)

display(success_lever_df)


,config_id,model,prompt_version,mode,trials,passed_trials,trial_pass_rate,cases,passed_cases,case_pass_rate
0,anthropic/claude-haiku-4.5 | v2 | parallel,anthropic/claude-haiku-4.5,v2,parallel,59,27,0.457627,41,21,0.512195
1,google/gemini-2.5-flash-lite | v1 | parallel,google/gemini-2.5-flash-lite,v1,parallel,59,10,0.169492,41,8,0.195122
2,google/gemini-2.5-flash-lite | v2 | parallel,google/gemini-2.5-flash-lite,v2,parallel,59,17,0.288136,41,13,0.317073
3,mistralai/mistral-small-2603 | v2 | parallel,mistralai/mistral-small-2603,v2,parallel,59,9,0.152542,41,8,0.195122
4,openai/gpt-5.6-luna | v2 | parallel,openai/gpt-5.6-luna,v2,parallel,59,35,0.593220,41,29,0.707317
5,qwen/qwen3.7-plus | v2 | parallel,qwen/qwen3.7-plus,v2,parallel,59,28,0.474576,41,23,0.560976


,model,before_prompt,before_passed_trials,before_trials,before_trial_pass_rate,after_prompt,after_passed_trials,after_trials,after_trial_pass_rate,change_percentage_points
0,google/gemini-2.5-flash-lite,v1,10,59,0.169492,v2,17,59,0.288136,11.864407


### H.2 — Turn-Count Lever: Controlled Strict-Sequential vs Parallel Replay


In [21]:
# The retained live battery contains parallel runs only. To avoid inventing a
# sequential live result, D2(c) is measured deterministically on the scripted
# trajectory set. Multi-action blocks are split into one action per model turn
# for the strict-sequential treatment. Cases, v2 prompt, tools, answer key,
# grader and the deployment step cap remain fixed.
#
# tiktoken counts are a reproducible proxy for the relative input/output growth
# caused by extra turns; they are not provider billing tokens.

from src.agent.loop import run_agent
from src.agent.prompt_loader import load_prompt
from src.backends.base import ModelBackend
from src.schemas import GuardConfig, ModelResponse, ToolResult
from src.tools.versioned import get_versioned_tool_registry
from Evals.scripted_cases import SCRIPTED_CASES
from Evals.evaluate import load_answer_key
from Evals.checks import evaluate_result

encoding = tiktoken.get_encoding("cl100k_base")


class CountingScriptedBackend:
    """Deterministic backend that adds proxy token counts to scripted replies."""

    name = "CountingScriptedBackend"

    def __init__(self, responses):
        self.responses = list(responses)
        self.index = 0

    def generate(self, messages, *, model: str, temperature: float = 0.0):
        if self.index >= len(self.responses):
            raise RuntimeError("SCRIPTED_BACKEND_EXHAUSTED")
        text = self.responses[self.index]
        self.index += 1
        compact_messages = json.dumps(
            messages,
            ensure_ascii=False,
            separators=(",", ":"),
        )
        return ModelResponse(
            text=text,
            model=model,
            tokens_in=len(encoding.encode(compact_messages)),
            tokens_out=len(encoding.encode(text)),
            cost_usd=0.0,
            latency_ms=0.0,
        )


def split_to_strict_sequential(responses):
    """Split every multi-action block into one-action model turns."""
    split = []
    for text in responses:
        obj = json.loads(text)
        if obj.get("type") == "action_block" and len(obj.get("actions", [])) > 1:
            for action in obj["actions"]:
                split.append(
                    json.dumps(
                        {
                            "type": "action_block",
                            "reasoning_summary": obj.get("reasoning_summary", ""),
                            "actions": [action],
                        },
                        ensure_ascii=False,
                        separators=(",", ":"),
                    )
                )
        else:
            split.append(text)
    return split


def replay_issue_decision_letter(**kwargs):
    """No-file-write replacement used only by the deterministic D2(c) replay."""
    return ToolResult(
        ok=True,
        data={
            "logged": True,
            "log_id": "D2C-REPLAY",
            "gate_result": "ACT_ALLOWED_REPLAY",
        },
    )


answer_key = {item["case_id"]: item for item in load_answer_key()}
replay_case_ids = sorted(set(SCRIPTED_CASES).intersection(answer_key))
replay_registry = get_versioned_tool_registry("v2")
replay_registry["issue_decision_letter"] = replay_issue_decision_letter
replay_prompt = load_prompt("v2")

turn_replay_rows = []

for case_id in replay_case_ids:
    expected = answer_key[case_id]["expected"]
    base_responses = SCRIPTED_CASES[case_id]

    for replay_mode in ["strict_sequential", "parallel"]:
        responses = (
            split_to_strict_sequential(base_responses)
            if replay_mode == "strict_sequential"
            else list(base_responses)
        )

        result = run_agent(
            case_id,
            backend=CountingScriptedBackend(responses),
            model="scripted-d2c-replay",
            parallel_tools=(replay_mode == "parallel"),
            autonomy="act",
            max_steps=DEPLOYMENT_STEP_CAP,
            budget_usd=0.0,
            guard_config=GuardConfig(),
            tool_registry=replay_registry,
            prompt_version="v2",
            system_prompt=replay_prompt,
            operator_approved=True,
        )

        l1 = evaluate_result(result, expected)
        proxy_layer1 = baseline_variable_cost(
            result.tokens_in,
            result.tokens_out,
            MODEL_PRICES[SELECTED_BASELINE_MODEL]["input_per_1m"],
            MODEL_PRICES[SELECTED_BASELINE_MODEL]["output_per_1m"],
        )

        turn_replay_rows.append(
            {
                "case_id": case_id,
                "replay_mode": replay_mode,
                "status": result.status,
                "turns": result.turns,
                "tool_calls": result.tool_calls,
                "proxy_tokens_in": result.tokens_in,
                "proxy_tokens_out": result.tokens_out,
                "proxy_layer1_usd_at_baseline_price": proxy_layer1,
                "code_passed": bool(l1["code_passed"]),
                "step_cap_hit": "STEP_LIMIT" in result.caps_fired,
            }
        )

turn_replay_detail_df = pd.DataFrame(turn_replay_rows)
turn_replay_summary_df = (
    turn_replay_detail_df
    .groupby("replay_mode", as_index=False)
    .agg(
        cases=("case_id", "nunique"),
        avg_turns=("turns", "mean"),
        median_turns=("turns", "median"),
        max_turns=("turns", "max"),
        avg_proxy_tokens_in=("proxy_tokens_in", "mean"),
        avg_proxy_tokens_out=("proxy_tokens_out", "mean"),
        avg_proxy_layer1_usd=("proxy_layer1_usd_at_baseline_price", "mean"),
        code_pass_rate=("code_passed", "mean"),
        step_cap_hits=("step_cap_hit", "sum"),
    )
)

display(turn_replay_summary_df)

seq_turn = turn_replay_summary_df.query("replay_mode == 'strict_sequential'").iloc[0]
par_turn = turn_replay_summary_df.query("replay_mode == 'parallel'").iloc[0]

LEVER_MEASUREMENTS["turn_count"].update(
    {
        "strict_sequential_avg_turns": seq_turn["avg_turns"],
        "parallel_avg_turns": par_turn["avg_turns"],
        "strict_sequential_proxy_input_tokens": seq_turn["avg_proxy_tokens_in"],
        "parallel_proxy_input_tokens": par_turn["avg_proxy_tokens_in"],
        "strict_sequential_pass_rate": seq_turn["code_pass_rate"],
        "parallel_pass_rate": par_turn["code_pass_rate"],
    }
)


,replay_mode,cases,avg_turns,median_turns,max_turns,avg_proxy_tokens_in,avg_proxy_tokens_out,avg_proxy_layer1_usd,code_pass_rate,step_cap_hits
0,parallel,41,4.097561,4.0,5,7744.365854,412.146341,0.002043,1.000000,0
1,strict_sequential,41,6.341463,6.0,8,12240.707317,429.073171,0.002963,0.878049,5


**Scope note.** This is a controlled **scripted replay**, not a second live
battery. It isolates the structural effect of collapsing independent tool calls
into one model turn while avoiding live-model variance and additional API spend.
The strict-sequential treatment splits multi-action blocks into one action per
model turn and keeps the v2 prompt, cases, tools, grader and 8-step deployment
cap fixed. Proxy token counts use `cl100k_base` consistently in both treatments.

This is more defensible than fabricating a missing live sequential run, but it
should be described as scripted D2(c) evidence in the report.


### H.3 — Observation-Size Lever: Full-Trial v1/v2 Compact-JSON Measurement


In [22]:
# Team measurement convention: observation tokens are the sum of the compact
# JSON observation messages sent back to the model within one trial. The same
# tiktoken encoding is applied to v1 and v2.

def load_result_artifact(model_id, prompt_version):
    matches = []
    for path in result_files:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        if data.get("model") == model_id and data.get("prompt_version") == prompt_version:
            matches.append((path, data))
    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one result for {model_id} {prompt_version}; found {len(matches)}"
        )
    return matches[0]


def compact_observation_trial_rows(path, data):
    rows = []
    for case in data["results"]:
        for trial in case["trials"]:
            run = trial["result"]
            by_turn = {}
            for item in run.get("tool_trace", []):
                by_turn.setdefault(int(item["turn"]), []).append(item["observation"])

            message_tokens = []
            message_chars = []
            for turn, observations in sorted(by_turn.items()):
                observations = sorted(observations, key=lambda x: x["call_id"])
                content = json.dumps(
                    {"type": "observations", "items": observations},
                    ensure_ascii=False,
                    separators=(",", ":"),
                )
                message_tokens.append(len(encoding.encode(content)))
                message_chars.append(len(content))

            rows.append(
                {
                    "source_file": path.name,
                    "model": data["model"],
                    "prompt_version": data["prompt_version"],
                    "case_id": case["case_id"],
                    "trial": trial["trial"],
                    "observation_messages": len(message_tokens),
                    "observation_tokens": sum(message_tokens),
                    "observation_characters": sum(message_chars),
                }
            )
    return rows


gemini_model = "google/gemini-2.5-flash-lite"
v1_path, v1_data = load_result_artifact(gemini_model, "v1")
v2_path, v2_data = load_result_artifact(gemini_model, "v2")

observation_trial_df = pd.DataFrame(
    compact_observation_trial_rows(v1_path, v1_data)
    + compact_observation_trial_rows(v2_path, v2_data)
)

v1_obs = observation_trial_df.query("prompt_version == 'v1'").copy()
v2_obs = observation_trial_df.query("prompt_version == 'v2'").copy()

observation_measurement_df = v1_obs.merge(
    v2_obs,
    on=["model", "case_id", "trial"],
    suffixes=("_v1", "_v2"),
    validate="one_to_one",
)
observation_measurement_df["token_change"] = (
    observation_measurement_df["observation_tokens_v2"]
    - observation_measurement_df["observation_tokens_v1"]
)
observation_measurement_df["pct_change"] = (
    observation_measurement_df["token_change"]
    / observation_measurement_df["observation_tokens_v1"].replace(0, pd.NA)
)

observation_summary_df = pd.DataFrame(
    [
        {
            "prompt_version": version,
            "trials": len(df),
            "total_observation_tokens": df["observation_tokens"].sum(),
            "avg_observation_tokens_per_trial": df["observation_tokens"].mean(),
            "median_observation_tokens_per_trial": df["observation_tokens"].median(),
            "avg_observation_messages_per_trial": df["observation_messages"].mean(),
        }
        for version, df in [("v1", v1_obs), ("v2", v2_obs)]
    ]
)

display(observation_summary_df)
display(observation_measurement_df.head())

v1_obs_summary = observation_summary_df.query("prompt_version == 'v1'").iloc[0]
v2_obs_summary = observation_summary_df.query("prompt_version == 'v2'").iloc[0]

LEVER_MEASUREMENTS["observation_size"].update(
    {
        "v1_avg_tokens_per_trial": v1_obs_summary["avg_observation_tokens_per_trial"],
        "v2_avg_tokens_per_trial": v2_obs_summary["avg_observation_tokens_per_trial"],
    }
)


,prompt_version,trials,total_observation_tokens,avg_observation_tokens_per_trial,median_observation_tokens_per_trial,avg_observation_messages_per_trial
0,v1,59,26080,442.033898,401.0,5.338983
1,v2,59,28069,475.745763,441.0,5.694915


,source_file_v1,model,prompt_version_v1,case_id,trial,observation_messages_v1,observation_tokens_v1,observation_characters_v1,source_file_v2,prompt_version_v2,observation_messages_v2,observation_tokens_v2,observation_characters_v2,token_change,pct_change
0,results__google-gemini-2-5-flash-lite__v1__par...,google/gemini-2.5-flash-lite,v1,CLM-8842,1,7,699,2512,results__google-gemini-2-5-flash-lite__v2__par...,v2,4,639,2379,-60,-0.085837
1,results__google-gemini-2-5-flash-lite__v1__par...,google/gemini-2.5-flash-lite,v1,CLM-8850,1,5,364,1308,results__google-gemini-2-5-flash-lite__v2__par...,v2,6,423,1502,59,0.162088
2,results__google-gemini-2-5-flash-lite__v1__par...,google/gemini-2.5-flash-lite,v1,CLM-8861,1,7,570,2034,results__google-gemini-2-5-flash-lite__v2__par...,v2,7,523,1885,-47,-0.082456
3,results__google-gemini-2-5-flash-lite__v1__par...,google/gemini-2.5-flash-lite,v1,CLM-8874,1,3,339,1205,results__google-gemini-2-5-flash-lite__v2__par...,v2,6,460,1628,121,0.356932
4,results__google-gemini-2-5-flash-lite__v1__par...,google/gemini-2.5-flash-lite,v1,CLM-8925,1,8,681,2540,results__google-gemini-2-5-flash-lite__v2__par...,v2,8,701,2607,20,0.029369


#### H.3.1 — Observation-Size Interpretation

Observation size is measured over the **full retained Gemini v1 and v2 trial
sets**, not only one pre-authorisation response. For each trial, the notebook
reconstructs exactly the compact JSON observation messages used by the loop,
counts them with the same `cl100k_base` tokenizer, and sums them once per trial.

This matches the team's agreed measurement convention. The result captures the
whole-run observation burden, including any path differences caused by the
v1/v2 treatment. The retained v1 and v2 runs use the same model but different
repository commits, so the commit mismatch remains a disclosed limitation.
Proxy tokens are used only for this relative comparison; provider billing tokens
come from the live result artifacts.


### H.4 — Tool-Block Size Comparison

In [23]:
from src.tools.base import ToolSpec
from src.tools.descriptors import get_tool_specs, render_tool_specs

encoding = tiktoken.get_encoding("cl100k_base")
final_specs = get_tool_specs("v2")

# D2(a) analytical baseline.
# The final tool set was consolidated from the outset, so the 'before'
# artifact is reconstructed only for token-cost comparison, using the
# documented design responsibilities and the same ToolSpec format.

reconstructed_lookup_member = ToolSpec(
    name="lookup_member",
    signature="lookup_member(member_id: str)",
    what="Retrieve the policy link for one member.",
    input_contract="member_id must be one non-empty member ID string.",
    return_contract=(
        "ToolResult. On success, data contains exactly one member with "
        "member_id and policy_id."
    ),
    fails_when=("INVALID_ARGUMENT", "MEMBER_NOT_FOUND"),
    irreversible=False,
)

reconstructed_lookup_policy = ToolSpec(
    name="lookup_policy",
    signature="lookup_policy(policy_id: str)",
    what="Retrieve policy status, dates, limits, and exclusions for one policy.",
    input_contract="policy_id must be one non-empty policy ID string.",
    return_contract=(
        "ToolResult. On success, data contains exactly one policy with policy_id, "
        "product, status, start_date, end_date, annual_limit, used_to_date, "
        "remaining_limit, and the complete exclusions list."
    ),
    fails_when=("INVALID_ARGUMENT", "POLICY_NOT_FOUND"),
    irreversible=False,
)

reconstructed_lookup_procedure = ToolSpec(
    name="lookup_procedure",
    signature="lookup_procedure(procedure_code: str)",
    what="Retrieve the reference facts for one procedure.",
    input_contract="procedure_code must be one non-empty procedure code string.",
    return_contract=(
        "ToolResult. On success, data contains procedure_code, description, "
        "and requires_preauth for exactly one procedure."
    ),
    fails_when=("INVALID_ARGUMENT", "PROCEDURE_NOT_FOUND"),
    irreversible=False,
)

reconstructed_check_coverage = ToolSpec(
    name="check_coverage",
    signature="check_coverage(member_id: str, procedure_code: str)",
    what="Check whether one procedure is excluded by the policy linked to one member.",
    input_contract="member_id and procedure_code must be non-empty strings.",
    return_contract=(
        "ToolResult. On success, data contains procedure_code, excluded, "
        "and exclusion_rule."
    ),
    fails_when=("INVALID_ARGUMENT", "NOT_FOUND"),
    irreversible=False,
)

reconstructed_check_required_documents = ToolSpec(
    name="check_required_documents",
    signature=(
        "check_required_documents(procedure_code: str, "
        "attached_documents: list[str])"
    ),
    what="Check the required document for one procedure and whether it is attached.",
    input_contract=(
        "procedure_code must be a non-empty string; attached_documents must be "
        "the claim's complete document list."
    ),
    return_contract=(
        "ToolResult. On success, data contains procedure_code, required_document, "
        "and document_present."
    ),
    fails_when=("INVALID_ARGUMENT", "PROCEDURE_NOT_FOUND"),
    irreversible=False,
)

reconstructed_before_specs = [
    final_specs["get_claim"],
    reconstructed_lookup_member,
    reconstructed_lookup_policy,
    reconstructed_lookup_procedure,
    reconstructed_check_coverage,
    reconstructed_check_required_documents,
    final_specs["get_preauthorisation"],
    final_specs["get_hospital_status"],
    final_specs["check_duplicate_claim"],
    final_specs["issue_decision_letter"],
]


def render_specs(specs):
    sections = []
    for spec in specs:
        sections.append(
            "\n".join(
                [
                    f"Name: {spec.name}",
                    f"Signature: {spec.signature}",
                    f"What: {spec.what}",
                    f"Input: {spec.input_contract}",
                    f"Returns: {spec.return_contract}",
                    f"Fails when: {', '.join(spec.fails_when)}",
                    f"Irreversible: {spec.irreversible}",
                ]
            )
        )
    return "\n\n".join(sections)


reconstructed_before_tool_block = render_specs(reconstructed_before_specs)
final_v2_tool_block = render_tool_specs("v2")

reconstructed_before_tokens = len(encoding.encode(reconstructed_before_tool_block))
final_v2_tool_block_tokens = len(encoding.encode(final_v2_tool_block))
reconstructed_before_tool_count = len(reconstructed_before_specs)
final_tool_count = len(final_specs)
tool_block_token_reduction = reconstructed_before_tokens - final_v2_tool_block_tokens
tool_block_reduction_pct = reduction_pct(
    reconstructed_before_tokens,
    final_v2_tool_block_tokens,
)

# Keep the v1 descriptor count only as a supplementary descriptor-rewrite
# diagnostic. It is not the D2(a) before value.
v1_tool_block_tokens = len(encoding.encode(render_tool_specs("v1")))

documented_consolidations = pd.DataFrame(
    [
        {
            "candidate_tool": "lookup_member",
            "decision": "merged/not exposed",
            "final_location": "lookup_policy",
            "reconstructed_baseline": "separate tool",
        },
        {
            "candidate_tool": "lookup_procedure",
            "decision": "merged/not exposed",
            "final_location": "check_coverage",
            "reconstructed_baseline": "separate tool",
        },
        {
            "candidate_tool": "check_required_documents",
            "decision": "merged/not exposed",
            "final_location": "check_coverage",
            "reconstructed_baseline": "separate tool",
        },
        {
            "candidate_tool": "web_search",
            "decision": "not added",
            "final_location": "local fixture data",
            "reconstructed_baseline": "excluded — not a consolidation reversal",
        },
    ]
)

tool_block_evidence_df = pd.DataFrame(
    [
        {
            "artifact": "reconstructed_pre_consolidation",
            "tool_count": reconstructed_before_tool_count,
            "cl100k_base_tokens": reconstructed_before_tokens,
            "evidence_status": "reconstructed from documented D2(a) consolidation decisions",
        },
        {
            "artifact": "final_consolidated_v2",
            "tool_count": final_tool_count,
            "cl100k_base_tokens": final_v2_tool_block_tokens,
            "evidence_status": "actual final repository descriptor block",
        },
    ]
)

assert reconstructed_before_tool_count == 10
assert final_tool_count == 7
assert reconstructed_before_tokens > final_v2_tool_block_tokens

print("Reconstructed before tool count:", reconstructed_before_tool_count)
print("Final consolidated tool count:", final_tool_count)
print("Reconstructed before tool-block tokens:", reconstructed_before_tokens)
print("Final v2 tool-block tokens:", final_v2_tool_block_tokens)
print("Token reduction:", tool_block_token_reduction)
print(f"Reduction: {tool_block_reduction_pct:.1%}")
print("Supplementary v1 descriptor tokens:", v1_tool_block_tokens)

display(tool_block_evidence_df)

LEVER_MEASUREMENTS["tool_block_size"].update(
    {
        "reconstructed_before_tokens": reconstructed_before_tokens,
        "final_v2_tokens": final_v2_tool_block_tokens,
        "reconstructed_before_tool_count": reconstructed_before_tool_count,
        "final_tool_count": final_tool_count,
        "token_reduction": tool_block_token_reduction,
        "reduction_pct": tool_block_reduction_pct,
    }
)

Reconstructed before tool count: 10
Final consolidated tool count: 7
Reconstructed before tool-block tokens: 1049
Final v2 tool-block tokens: 841
Token reduction: 208
Reduction: 19.8%
Supplementary v1 descriptor tokens: 809


,artifact,tool_count,cl100k_base_tokens,evidence_status
0,reconstructed_pre_consolidation,10,1049,reconstructed from documented D2(a) consolidat...
1,final_consolidated_v2,7,841,actual final repository descriptor block


#### H.4.1 — Interpretation

The final tool set was consolidated from the outset, so no historical
pre-consolidation implementation exists. For D6, the notebook therefore
constructs an analytical pre-consolidation baseline from the documented
D2(a) design responsibilities and compares it with the final tool block
using the same descriptor format and `cl100k_base` tokenizer.

The comparison is used only to estimate the token effect of consolidation;
it is not presented as a historical repository version.

### H.5 — Final Four-Lever Evidence Status

The four D6 levers are covered as follows:

- **Tool block:** analytical pre-consolidation baseline → final consolidated block.
- **Turn count:** controlled scripted sequential → parallel comparison.
- **Observation size:** retained Gemini v1 → v2 live traces using the agreed compact-JSON convention.
- **Success rate:** retained Gemini v1 → v2 live evaluation results.

Evidence scope is stated in the exported ledger and report draft.

In [24]:
lever_status_df = lever_measurement_status(
    LEVER_MEASUREMENTS
)

display(lever_status_df)

,lever,status,source
0,tool_block_size,READY_WITH_RECONSTRUCTION_NOTE,D2(a) documented consolidation + reconstructed...
1,turn_count,READY_WITH_SCOPE_NOTE,Controlled scripted D2(c) replay
2,observation_size,READY_WITH_COMMIT_NOTE,Retained Gemini v1/v2 live traces; compact JSO...
3,success_rate,READY_WITH_COMMIT_NOTE,D4/D5 retained Gemini v1/v2 live results


## Section I — Final Output and Export Structure

The notebook exports the cost ledger, three-layer model, sensitivity,
break-even analysis, deployment caps and four-lever evidence to
`outputs/cost/`. Layer 3 is set to the explicit prototype assumption
of US$0/month.

Where evidence is not a direct historical/live comparison, its scope is
stated explicitly rather than treated as equivalent evidence.

### I.1 — Final Cost-Model Export

In [25]:
from pathlib import Path

FINAL_OUTPUT_DIR = Path("outputs/cost")
FINAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Deployment caps and explicit modelling assumptions.
deployment_caps_df = pd.DataFrame(
    [
        {"cap": "step_cap", "value": DEPLOYMENT_STEP_CAP, "unit": "model turns/run", "basis": "team spec + selected Luna max observed turn count"},
        {"cap": "per_run_budget_ceiling", "value": PER_RUN_BUDGET_CEILING_USD, "unit": "USD/run", "basis": "team spec"},
        {"cap": "monthly_per_user_limit", "value": MONTHLY_RUN_LIMIT_PER_USER, "unit": "agent runs/user/month", "basis": "deployment assumption"},
        {"cap": "implied_monthly_user_budget_ceiling", "value": MONTHLY_USER_BUDGET_CEILING_USD, "unit": "USD/user/month", "basis": "200 runs × US$0.05/run"},
    ]
)

assumptions_df = pd.DataFrame(
    [
        {"assumption": "monthly_volume", "value": MONTHLY_VOLUME, "unit": "claims/month", "source_or_status": "Problem A"},
        {"assumption": "assessor_rate", "value": ASSESSOR_USD_PER_HOUR, "unit": "USD/hour", "source_or_status": "Problem A"},
        {"assumption": "minutes_per_escalation", "value": MINUTES_PER_ESCALATION, "unit": "minutes", "source_or_status": "Problem A"},
        {"assumption": "failure_cost", "value": FAILURE_COST_USD, "unit": "USD/failure", "source_or_status": "derived"},
        {"assumption": "layer3_fixed_monthly", "value": FIXED_MONTHLY_USD, "unit": "USD/month", "source_or_status": "explicit prototype-scope team assumption"},
    ]
)

# Compact cost-lever ledger for the report.
seq_turn = turn_replay_summary_df.query("replay_mode == 'strict_sequential'").iloc[0]
par_turn = turn_replay_summary_df.query("replay_mode == 'parallel'").iloc[0]
obs_v1 = observation_summary_df.query("prompt_version == 'v1'").iloc[0]
obs_v2 = observation_summary_df.query("prompt_version == 'v2'").iloc[0]

lever_export_df = pd.DataFrame(
    [
        {
            "lever": "tool_block_size",
            "status": "READY_WITH_RECONSTRUCTION_NOTE",
            "before": reconstructed_before_tokens,
            "after": final_v2_tool_block_tokens,
            "unit": "cl100k_base proxy tokens",
            "measured_effect": (
                f"-{tool_block_token_reduction} tokens ({tool_block_reduction_pct:.1%} reduction)"
            ),
            "evidence_type": "analytical D2(a) baseline → actual final block",
        },
        {
            "lever": "turn_count",
            "status": "READY_WITH_SCOPE_NOTE",
            "before": seq_turn["avg_turns"],
            "after": par_turn["avg_turns"],
            "unit": "average turns/case",
            "measured_effect": (
                f"proxy input tokens {seq_turn['avg_proxy_tokens_in']:.1f} → {par_turn['avg_proxy_tokens_in']:.1f}; "
                f"code pass rate {seq_turn['code_pass_rate']:.1%} → {par_turn['code_pass_rate']:.1%}; "
                f"strict-sequential step-cap hits {int(seq_turn['step_cap_hits'])}"
            ),
            "evidence_type": "controlled scripted D2(c) replay",
        },
        {
            "lever": "observation_size",
            "status": "READY_WITH_COMMIT_NOTE",
            "before": obs_v1["avg_observation_tokens_per_trial"],
            "after": obs_v2["avg_observation_tokens_per_trial"],
            "unit": "avg cl100k_base observation tokens/trial",
            "measured_effect": (
                f"total tokens {int(obs_v1['total_observation_tokens'])} → {int(obs_v2['total_observation_tokens'])}"
            ),
            "evidence_type": "retained Gemini live v1/v2 traces",
        },
        {
            "lever": "success_rate",
            "status": "READY_WITH_COMMIT_NOTE",
            "before": success_lever_df.iloc[0]["before_trial_pass_rate"],
            "after": success_lever_df.iloc[0]["after_trial_pass_rate"],
            "unit": "trial pass rate",
            "measured_effect": f"{success_lever_df.iloc[0]['change_percentage_points']:+.2f} percentage points",
            "evidence_type": "retained Gemini live v1/v2 results",
        },
    ]
)

exports = {
    "cost_ledger.csv": final_raw_ledger_df,
    "model_eval_summary.csv": model_eval_summary_df,
    "model_cost_summary.csv": model_cost_df,
    "sensitivity.csv": final_sensitivity_df,
    "break_even.csv": formal_break_even_df,
    "turn_replay_detail.csv": turn_replay_detail_df,
    "turn_replay_summary.csv": turn_replay_summary_df,
    "observation_measurement.csv": observation_measurement_df,
    "observation_summary.csv": observation_summary_df,
    "tool_block_evidence.csv": tool_block_evidence_df,
    "cost_lever_ledger.csv": lever_export_df,
    "deployment_caps.csv": deployment_caps_df,
    "assumptions.csv": assumptions_df,
}

for filename, df in exports.items():
    path = FINAL_OUTPUT_DIR / filename
    df.to_csv(path, index=False)
    print(f"Saved: {path} ({len(df)} rows)")

# Preserve the two tool-block artifacts that underpin the D2(a) comparison.
(FINAL_OUTPUT_DIR / "tool_block_before_reconstructed.txt").write_text(
    reconstructed_before_tool_block, encoding="utf-8"
)
(FINAL_OUTPUT_DIR / "tool_block_after_final_v2.txt").write_text(
    final_v2_tool_block, encoding="utf-8"
)
print("Saved reconstructed/final tool-block text artifacts")

# ------------------------------------------------------------------
# Report-ready D6 summary generated from the same notebook variables.
# ------------------------------------------------------------------
baseline_row = model_cost_df[
    (model_cost_df["model"] == SELECTED_BASELINE_MODEL)
    & (model_cost_df["prompt_version"] == "v2")
].iloc[0]

gemini_v1_cost = model_cost_df[
    (model_cost_df["model"] == "google/gemini-2.5-flash-lite")
    & (model_cost_df["prompt_version"] == "v1")
].iloc[0]
gemini_v2_cost = model_cost_df[
    (model_cost_df["model"] == "google/gemini-2.5-flash-lite")
    & (model_cost_df["prompt_version"] == "v2")
].iloc[0]

baseline_sens = sensitivity_summary_df[
    sensitivity_summary_df["configuration"] == baseline_row["config_id"]
].iloc[0]

be_row = formal_break_even_df.iloc[0]

summary_md = f"""# D6 Final Cost Summary — LIANG LUWEN

- Selected baseline: `{SELECTED_BASELINE_MODEL}` v2
- Measured trial success: {int(baseline_row['passed_trials'])}/{int(baseline_row['trials'])} = {baseline_row['trial_pass_rate']:.2%}
- Layer 1 variable cost: US${baseline_row['layer1_cost_per_trial_usd']:.6f}/task
- Layer 2 expected fallback: US${baseline_row['layer2_fallback_cost_per_trial_usd']:.6f}/task
- Cost-to-serve: US${baseline_row['cost_to_serve_per_trial_usd']:.6f}/task
- Layer 3: US${FIXED_MONTHLY_USD:.2f}/month (prototype-scope assumption)
- Monthly cost at {MONTHLY_VOLUME:,} claims: US${baseline_row['monthly_all_in_usd']:,.2f}
- Sensitivity (-10pp / measured / +10pp): US${baseline_sens['low_cost_to_serve_usd']:.4f} / US${baseline_sens['measured_cost_to_serve_usd']:.4f} / US${baseline_sens['high_cost_to_serve_usd']:.4f}
- Gemini v2 break-even success rate vs Luna: {be_row['break_even_success_rate']:.2%}; measured Gemini v2 = {be_row['cheap_measured_success_rate']:.2%}
- Deployment caps: {DEPLOYMENT_STEP_CAP} steps/run; US${PER_RUN_BUDGET_CEILING_USD:.2f}/run; {MONTHLY_RUN_LIMIT_PER_USER} runs/user/month (implied US${MONTHLY_USER_BUDGET_CEILING_USD:.2f}/user/month ceiling)

## Cost-lever evidence

{lever_export_df.to_markdown(index=False)}

## Disclosed limitations

1. D2(a) has no historical pre-consolidation implementation; its before value is an analytical baseline constructed from the documented design responsibilities.
2. D2(c) turn-count evidence is a controlled scripted replay, not a second live-model battery.
3. The retained Gemini v1 control uses commit `{V1_CONTROL_COMMIT}` while final v2 uses `{V2_FROZEN_COMMIT}`.
"""
(FINAL_OUTPUT_DIR / "D6_COST_SUMMARY.md").write_text(summary_md, encoding="utf-8")

section4_md = f"""## 4 What It Costs

### 4.1–4.2 Baseline and three-layer model
The selected baseline is `{SELECTED_BASELINE_MODEL}` v2, which had the lowest measured cost-to-serve among the five v2 battery configurations. Across {int(baseline_row['trials'])} trials it used {int(baseline_row['tokens_in']):,} input and {int(baseline_row['tokens_out']):,} output tokens and passed {int(baseline_row['passed_trials'])}/{int(baseline_row['trials'])} trials ({baseline_row['trial_pass_rate']:.2%}). At list prices of US${baseline_row['input_price_per_1m']:.2f}/1M input and US${baseline_row['output_price_per_1m']:.2f}/1M output tokens, Layer 1 is US${baseline_row['layer1_cost_per_trial_usd']:.6f} per task. Problem A prices one human escalation at US$38/hour × 12 minutes = US${FAILURE_COST_USD:.2f}; therefore Layer 2 is (1 − {baseline_row['trial_pass_rate']:.4f}) × US${FAILURE_COST_USD:.2f} = US${baseline_row['layer2_fallback_cost_per_trial_usd']:.4f}. Total cost-to-serve is US${baseline_row['cost_to_serve_per_trial_usd']:.4f} per task. Layer 3 is explicitly assumed to be US$0/month for the submitted prototype because no dedicated fixed infrastructure, licence, monitoring, storage or maintenance charge is documented. At {MONTHLY_VOLUME:,} claims/month, total monthly cost is US${baseline_row['monthly_all_in_usd']:,.2f}. This US$0 is a prototype assumption, not a production operating-cost estimate.

### 4.3 Four cost levers
The cost ledger separates evidence by scope. Because D2(a) was consolidated from the outset, the notebook uses an analytical pre-consolidation baseline constructed from the documented design responsibilities. With the same descriptor format and `cl100k_base` tokenizer, tool-block size changes from {reconstructed_before_tokens} to {final_v2_tool_block_tokens} tokens, a reduction of {tool_block_token_reduction} tokens ({tool_block_reduction_pct:.1%}). The baseline is used only for cost comparison and is not presented as a historical implementation. For turn count, a controlled scripted D2(c) replay keeps the v2 prompt, cases, tools, grader and 8-step cap fixed: strict sequential execution averaged {seq_turn['avg_turns']:.2f} turns versus {par_turn['avg_turns']:.2f} in parallel, while code pass rate moved from {seq_turn['code_pass_rate']:.1%} to {par_turn['code_pass_rate']:.1%}; strict sequential also hit the step cap in {int(seq_turn['step_cap_hits'])} cases. Proxy input tokens moved from {seq_turn['avg_proxy_tokens_in']:.1f} to {par_turn['avg_proxy_tokens_in']:.1f} per case. Observation size is measured over all paired Gemini v1/v2 live trials using the sum of compact JSON observation messages per trial: {obs_v1['avg_observation_tokens_per_trial']:.1f} → {obs_v2['avg_observation_tokens_per_trial']:.1f} proxy tokens/trial. The retained Gemini success-rate comparison improved from {success_lever_df.iloc[0]['before_trial_pass_rate']:.2%} to {success_lever_df.iloc[0]['after_trial_pass_rate']:.2%} (+{success_lever_df.iloc[0]['change_percentage_points']:.2f}pp), reducing cost-to-serve by US${gemini_v1_cost['cost_to_serve_per_trial_usd'] - gemini_v2_cost['cost_to_serve_per_trial_usd']:.4f}/task. The v1/v2 commit mismatch is disclosed.

### 4.4–4.5 Sensitivity, break-even and caps
For Luna, ±10 percentage points around the measured success rate moves cost-to-serve from US${baseline_sens['low_cost_to_serve_usd']:.4f} at the lower bound to US${baseline_sens['high_cost_to_serve_usd']:.4f} at the upper bound, showing that fallback dominates token price. Gemini v2 needs {be_row['break_even_success_rate']:.2%} success to match Luna's measured cost-to-serve but achieved {be_row['cheap_measured_success_rate']:.2%}. Deployment caps are {DEPLOYMENT_STEP_CAP} steps/run, US${PER_RUN_BUDGET_CEILING_USD:.2f}/run and {MONTHLY_RUN_LIMIT_PER_USER} runs/user/month (an implied US${MONTHLY_USER_BUDGET_CEILING_USD:.2f}/user/month maximum under the per-run ceiling).
"""
(FINAL_OUTPUT_DIR / "section4_report_draft.md").write_text(section4_md, encoding="utf-8")

print("Saved report-ready markdown files in", FINAL_OUTPUT_DIR)

Saved: outputs/cost/cost_ledger.csv (354 rows)
Saved: outputs/cost/model_eval_summary.csv (6 rows)
Saved: outputs/cost/model_cost_summary.csv (6 rows)
Saved: outputs/cost/sensitivity.csv (30 rows)
Saved: outputs/cost/break_even.csv (1 rows)
Saved: outputs/cost/turn_replay_detail.csv (82 rows)
Saved: outputs/cost/turn_replay_summary.csv (2 rows)
Saved: outputs/cost/observation_measurement.csv (59 rows)
Saved: outputs/cost/observation_summary.csv (2 rows)
Saved: outputs/cost/tool_block_evidence.csv (2 rows)
Saved: outputs/cost/cost_lever_ledger.csv (4 rows)
Saved: outputs/cost/deployment_caps.csv (4 rows)
Saved: outputs/cost/assumptions.csv (5 rows)
Saved reconstructed/final tool-block text artifacts
Saved report-ready markdown files in outputs/cost


In [26]:
print("Final export files:\n")

for path in sorted(FINAL_OUTPUT_DIR.glob("*.csv")):
    print(
        f"{path.name:<45} "
        f"{path.stat().st_size / 1024:.1f} KB"
    )


Final export files:

assumptions.csv                               0.3 KB
break_even.csv                                0.4 KB
cost_ledger.csv                               99.3 KB
cost_lever_ledger.csv                         0.8 KB
deployment_caps.csv                           0.3 KB
model_cost_summary.csv                        3.4 KB
model_eval_summary.csv                        1.8 KB
observation_measurement.csv                   11.9 KB
observation_summary.csv                       0.3 KB
sensitivity.csv                               3.5 KB
tool_block_evidence.csv                       0.2 KB
turn_replay_detail.csv                        5.9 KB
turn_replay_summary.csv                       0.4 KB


## Section J — Final Data Validation

The final validation checks the expected six result configurations, trial
counts, live backend, unique run IDs, token fields, pricing, Layer 3,
deployment caps and the four cost-lever outputs.

Methodological scope notes are disclosed in the generated summary and
report draft and are not hidden by the validation checks.

### J.1 — Final Live-Data Validation

In [27]:
# Derive expected counts from the source artifacts.
expected_trials = 0
for path in result_files:
    with open(path, "r", encoding="utf-8") as f:
        source_data = json.load(f)
    expected_trials += source_data["summary"]["total_trials"]

validation_checks = {}
required_columns = [
    "run_id", "case_id", "model", "prompt_version", "backend", "mode",
    "passed", "turns", "tool_calls", "tokens_in", "tokens_out",
    "latency_ms", "config_id",
]

validation_checks["required_columns_present"] = all(
    col in final_raw_ledger_df.columns for col in required_columns
)
validation_checks["six_final_result_files"] = len(result_files) == EXPECTED_FINAL_RESULTS
validation_checks["five_v2_one_v1"] = (
    (model_eval_summary_df["prompt_version"] == "v2").sum() == 5
    and (model_eval_summary_df["prompt_version"] == "v1").sum() == 1
)
validation_checks["row_count_matches_source_summaries"] = (
    len(final_raw_ledger_df) == expected_trials == 354
)
validation_checks["six_unique_configurations"] = final_raw_ledger_df["config_id"].nunique() == 6
validation_checks["run_ids_unique"] = final_raw_ledger_df["run_id"].is_unique
validation_checks["all_trial_backends_live"] = set(final_raw_ledger_df["backend"].dropna()) == {"LiveBackend"}
validation_checks["no_missing_pass_result"] = final_raw_ledger_df["passed"].notna().all()
validation_checks["tokens_nonnegative"] = (
    (final_raw_ledger_df["tokens_in"] >= 0).all()
    and (final_raw_ledger_df["tokens_out"] >= 0).all()
)
validation_checks["all_live_results_parallel"] = set(final_raw_ledger_df["mode"].dropna()) == {"parallel"}
validation_checks["all_models_have_prices"] = all(
    model in MODEL_PRICES for model in final_raw_ledger_df["model"].unique()
)
validation_checks["all_use_gpt_4_1_mini_judge"] = all(
    "openai/gpt-4.1-mini" in path.read_text(encoding="utf-8") for path in result_files
)
validation_checks["layer3_resolved"] = FIXED_MONTHLY_USD == 0.0
validation_checks["deployment_caps_defined"] = (
    DEPLOYMENT_STEP_CAP == 8
    and PER_RUN_BUDGET_CEILING_USD == 0.05
    and MONTHLY_RUN_LIMIT_PER_USER == 200
)
validation_checks["d2c_replay_has_both_modes"] = set(turn_replay_summary_df["replay_mode"]) == {"strict_sequential", "parallel"}
validation_checks["observation_pairs_complete"] = len(observation_measurement_df) == 59
validation_checks["tool_block_reconstruction_ready"] = (
    reconstructed_before_tool_count == 10
    and final_tool_count == 7
    and reconstructed_before_tokens > final_v2_tool_block_tokens > 0
)

validation_df = pd.DataFrame(
    [{"check": check, "passed": bool(passed)} for check, passed in validation_checks.items()]
)

display(validation_df)
print("\nOverall validation:", "PASS" if validation_df["passed"].all() else "FAIL")


,check,passed
0,required_columns_present,True
1,six_final_result_files,True
2,five_v2_one_v1,True
3,row_count_matches_source_summaries,True
4,six_unique_configurations,True
5,run_ids_unique,True
6,all_trial_backends_live,True
7,no_missing_pass_result,True
8,tokens_nonnegative,True
9,all_live_results_parallel,True



Overall validation: PASS


In [28]:
VALIDATION_PATH = FINAL_OUTPUT_DIR / "validation.csv"

validation_df.to_csv(
    VALIDATION_PATH,
    index=False,
)

print("Saved:", VALIDATION_PATH)
print("Exists:", VALIDATION_PATH.exists())


Saved: outputs/cost/validation.csv
Exists: True


## Section K — Handoff Package

After **Runtime → Restart session and run all**, the final D6 artifacts are in
`outputs/cost/`. The notebook also creates a single ZIP below for easy handoff.
The GitHub repository remains the formal source of truth; the ZIP is only a
transfer convenience.

In [29]:
import shutil

HANDOFF_ARCHIVE = Path("outputs/liang_luwen_d6_cost_outputs")
archive_path = shutil.make_archive(
    str(HANDOFF_ARCHIVE),
    "zip",
    root_dir=FINAL_OUTPUT_DIR,
)
print("Created:", archive_path)
print("Cost notebook validation:", "PASS" if validation_df["passed"].all() else "FAIL")

Created: /content/A2_PE6201_GRP4/outputs/liang_luwen_d6_cost_outputs.zip
Cost notebook validation: PASS
